In [ ]:
!pip install av==11.0.0
import torch
import torchvision
import os
import re
from google.colab import drive
import math
from torchvision.transforms import v2
import random
! if [ -d pertwee ]; then (cd pertwee; git pull); else git clone https://github.com/occipita/pertwee.git; fi
import pertwee
import importlib
importlib.reload(pertwee) # perwee may have changed due to a "git pull" operation above, so reload it just in case
importlib.reload(pertwee.normalisations)
importlib.reload(pertwee.frameutils)

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
Unpacking objects: 100% (3/3), 292 bytes | 292.00 KiB/s, done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
From https://github.com/occipita/pertwee
   8f168ba..b80824c  main       -> origin/main
Updating 8f168ba..b80824c
Fast-forward
 frameutils.py | 1 +
 1 file changed, 1 insertion(+)


<module 'pertwee.frameutils' from '/content/pertwee/frameutils.py'>

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
#
# global variables that define behaviour
#

# file selection
driveLoc = '/content/drive'
vpath = driveLoc + "/MyDrive/videos"
videoExts = [".mp4",".mkv",".avi"]
outpath = driveLoc + "/MyDrive/trainingdata"

# target resolution for output images
expectedHRes = 352
expectedVRes = 288
# the number of frames encoded in each image of each output sequence
framesToMerge = [2,1,1,2,3] # total = 3 frames before, 6 frames after transition
# number of frames to begin the output sequence prior to point listed in transitions data file
# probably best if this occurs between two "1"s in framesToMerge.
startFramesBeforeTransition = 3

# ratio of transition to non-transition frames to include in training and validation sets
nonTransitionRatio = 3
# chance to allocate frame to validation set
validationProbability = 0.125




In [ ]:
# FIXME - need a change here to identify items that should specifically be included as
# negative results (e.g. previous false positives, frames that were spotted as containing
# unusual changes that don't constitute a transition, etc).
def readTransitionsData (tpath):
    with open(tpath) as f:
      return [
          [s for s in re.split ("[- ]", line.strip()) if len(s)>0]
          for line in f.readlines()
      ]



In [ ]:
if os.path.exists(driveLoc):
  print ("Drive already mounted")
else:
  print ("Mounting drive")
  drive.mount(driveLoc)

datafiles = os.listdir(vpath)
for f in datafiles:
  fullPath = vpath+"/"+f
  splitext = os.path.splitext(f)
  if not os.path.isfile(fullPath) or splitext[1] not in videoExts:
    print ("Skipped (not a video): ",f)
    continue
  fullPathTransitions = vpath + "/" + splitext[0] + ".dat"
  if not os.path.isfile(fullPathTransitions):
    print ("Skipped (no transitions data): ", f)
    continue

  trainingOutputPath = outpath + f"/anglechange-training-{splitext[0]}.dat"
  validationOutputPath = outpath + f"/anglechange-validation-{splitext[0]}.dat"
  if os.path.isfile(trainingOutputPath) and os.path.isfile(validationOutputPath):
    print ("Skipped (all outputs already exist): ", f)
    continue

  print (f)

  video = torchvision.io.VideoReader(fullPath, "video")
  videoMetadata = video.get_metadata()
  fps = videoMetadata["video"]["fps"][0]
  print (videoMetadata)

  # for some reason, the metadata doesn't include number of channels or video resolution (!)
  # so work this out from the first frame:

  firstFrame = next(video)
  shape = firstFrame['data'].shape;
  print (shape)
  normalisers = pertwee.normalisations.defaultNormalisers(shape)

  transitions = readTransitionsData (fullPathTransitions)
  counter = 0

  usedFrameTimes = set()
  lastTransitionTime = 0  # track last defined transition because video may contain additional transitions after this

  # lists of pairs of input and expected result tensors to collect the generated data items in:
  trainingData = []
  validationData = []
  outTransition = torch.tensor(1, dtype=torch.float16)
  outNonTransition = torch.tensor(0, dtype=torch.float16)

  def addDataItem (inputs, outputs):
    val = (inputs, outputs)
    if random.random() < validationProbability:
      validationData.append(val)
    else:
      trainingData.append(val)

  # First generate output data for the transitions:
  rows = len(normalisers)
  cols = len(framesToMerge)
  #fig = plt.figure (figsize=(rows*2.4,cols*3))
  for transition in transitions:
    if len(transition) > 1:  # if there's a range and/or a description, it's not a regular transition, so we don't want to include it
      continue

    transitionTime = pertwee.frameutils.timestampToSeconds (transition[0], fps)
    usedFrameTimes.add (transitionTime)
    lastTransitionTime = transitionTime
    inputTensor = pertwee.frameutils.loadFrames (video, transitionTime, fps, startFramesBeforeTransition, framesToMerge)
    for normalise in normalisers:
      normalised = normalise(inputTensor)
      addDataItem (normalised, outTransition)
      #for col in range(0,cols):
      #  fig.add_subplot(rows,cols,counter*cols+col + 1)
      #  plt.imshow(v2.ToPILImage()(normalised[col]), cmap='gray', vmin=0, vmax=255)
      #counter += 1

  # Now generate some non-transition data:
  nonTransitionCount = len(trainingData)*nonTransitionRatio
  print (f"\n\nGenerating {nonTransitionCount} non-transition training items from frames before: {lastTransitionTime}")
  #rows = 10
  #cols = len(framesToMerge)
  #fig = plt.figure (figsize=(rows*2.4,cols*3))
  # FIXME: focus on non-transitions identified in transition data

  while nonTransitionCount > 0:
    fTime = -1
    while fTime < 0 or fTime in usedFrameTimes:
      fTime = pertwee.frameutils.randomFrameTime (fps, lastTransitionTime)
    #usedFrameTimes.add (fTime)
    inputTensor = pertwee.frameutils.loadFrames (video, fTime, fps, startFramesBeforeTransition, framesToMerge)
    normalise = random.choice(normalisers)
    normalised = normalise(inputTensor)

    # FIXME we also want to be able to produce alternative transformations that pull small sections of larger images!
    # FIXME also consider augmentation strategies

    addDataItem (normalised, outNonTransition)
    nonTransitionCount -= 1

    #for col in range(0,cols):
    #  fig.add_subplot(rows,cols,counter*cols+col + 1)
    #  plt.imshow(v2.ToPILImage()(normalised[col]), cmap='gray', vmin=0, vmax=255)

  print (f"\n\nTraining sets generated ({len(trainingData)} training items, {len(validationData)} validation items)")

  torch.save(trainingData, trainingOutputPath)
  torch.save(validationData, validationOutputPath)

  print ("Training sets saved")
    #counter += 1
    #if counter >= rows:
    #  break


Drive already mounted
Skipped (no transitions data):  dw06e03p3.avi
dw06e03p2.avi
{'video': {'fps': [25.0], 'duration': [1468.04]}, 'audio': {'framerate': [48000.0], 'duration': [1468.032]}}
torch.Size([3, 528, 704])


Generating 3780 non-transition training items from frames before: 735.56


Training sets generated (4579 training items, 644 validation items)
Training sets saved
Skipped (not a video):  dw06e03p2.dat
